# Q2A | Summarize the Data

> *Summarize the data. How much data is present? What attributes/features are continuous valued? Which attributes are categorical?*

This notebook answers the question in three parts:

1. **How much data is present?** Rows, columns, missing values, and duplicates.
2. **Which attributes are continuous?** Data types, distinct values, value ranges, and the date column.
3. **Which attributes are categorical?** The values each categorical attribute takes, and the one column that is neither.

---

#### **IMPORTANT**

The dataset is a single file with no `train` / `test` split. The split is made in a later notebook, so this summary describes the whole dataset.

# Answer Summary

- **How much data is present?** 731 rows and 16 columns (the target `Rentals` and 15 others), with no missing values and no duplicate rows. Each row is one day, from 2011-01-01 to 2012-12-31.
- **Which attributes are continuous?** 8: `Temp`, `FeelsLike`, `Humidity`, and `Wind` (decimals between 0 and 1), `Casual`, `Registered`, and `Rentals` (whole-number counts with 606 to 696 distinct values), and `Date` (one value per day).
- **Which attributes are categorical?** 7: `Season`, `Year`, `Month`, `IsHoliday`, `Weekday`, `IsWorkday`, and `Weather`, each with 2 to 12 distinct values. `ID` is neither: it numbers the rows.

**Carry forward:**

- The rows are in date order, one per day with no gaps, which matters for how the data is split.
- `Date` is not stored as a number, and `ID` carries the same information as `Date`.
- `Casual` and `Registered` add up to `Rentals`, which matters for whether they can be used as attributes.
- The categorical attributes are stored as integer codes, and `IsHoliday` and `Weather` each have a value with only 21 rows.
- The scales differ: 0 to 1 for the measurements, up to 12 for the codes, thousands for the counts.

---

# Imports

In [1]:
import pandas as pd

from Q2.consts import *

from utils import read_csv_to_df

# Fetch Data

In [2]:
df = read_csv_to_df(path=DATA_PATH, parse_dates=["dteday"])
df = df.rename(columns=SHORT_NAMES)

# 1. How Much Data is Present?

In [3]:
summary = pd.DataFrame(
	{
		"row count": [ df.shape[0] ],
		"columns count": [ df.shape[1] ],
		"missing values": [ df.isnull().sum().sum() ],
		"rows w/ missing values": [ df.isnull().any(axis=1).sum() ],
		"% rows w/ missing values": [
			round(df.isnull().any(axis=1).sum() / df.shape[0] * 100, 2),
		]
	},
	index=["Full Dataset"],
)
s = summary.T
display(s)

df.info()

,Full Dataset
row count,731.0
columns count,16.0
missing values,0.0
rows w/ missing values,0.0
% rows w/ missing values,0.0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 731 entries, 0 to 730
Data columns (total 16 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   ID          731 non-null    int64         
 1   Date        731 non-null    datetime64[ns]
 2   Season      731 non-null    int64         
 3   Year        731 non-null    int64         
 4   Month       731 non-null    int64         
 5   IsHoliday   731 non-null    int64         
 6   Weekday     731 non-null    int64         
 7   IsWorkday   731 non-null    int64         
 8   Weather     731 non-null    int64         
 9   Temp        731 non-null    float64       
 10  FeelsLike   731 non-null    float64       
 11  Humidity    731 non-null    float64       
 12  Wind        731 non-null    float64       
 13  Casual      731 non-null    int64         
 14  Registered  731 non-null    int64         
 15  Rentals     731 non-null    int64         
dtypes: datetime64[ns](1), floa

- 16 columns: the target `Rentals` and 15 others. No value is missing, so there is nothing to impute.
- `Date` is the only non-numeric column (`datetime64`).

## Duplicate Rows

In [4]:
df.duplicated().sum()

0

- No duplicate rows, so there is nothing to remove.

# 2. Which Attributes are Continuous?

An attribute is continuous if it measures an amount that can take many values within a range. The stored data type does not settle this, so the evidence is the number of distinct values and the range of each column.

In [5]:
info_df = pd.DataFrame({
	"dtype": df.dtypes,
	"non-null values": df.count(),
	"unique values": df.nunique(),
	"most frequent value count": df.apply(lambda col: col.value_counts().iloc[0]),
}).sort_values(by=["unique values", "most frequent value count"], ascending=False)

info_df

,dtype,non-null values,unique values,most frequent value count
ID,int64,731,731,1
Date,datetime64[ns],731,731,1
Rentals,int64,731,696,2
FeelsLike,float64,731,690,4
Registered,int64,731,679,3
Wind,float64,731,650,3
Casual,int64,731,606,4
Humidity,float64,731,595,4
Temp,float64,731,499,5
Month,int64,731,12,62


- The distinct value counts split the 16 columns into two groups with nothing in between: 7 columns with 2 to 12 distinct values, and 9 columns with 499 to 731 where no value repeats more than 5 times.
- All 7 columns in the first group are `int64`, like the counts `Casual`, `Registered`, and `Rentals`, so the data type does not tell the two groups apart.

## Value Ranges

In [6]:
numeric_df = df.select_dtypes(include="number")

value_ranges = pd.DataFrame({
	"min": numeric_df.min().round(3),
	"max": numeric_df.max().round(3),
	"mean": numeric_df.mean().round(3),
	"std": numeric_df.std().round(3),
	"whole numbers only": numeric_df.apply(lambda col: (col.dropna() % 1 == 0).all()),
}, index=numeric_df.columns)

value_ranges

,min,max,mean,std,whole numbers only
ID,1.000,731.000,366.000,211.166,True
Season,1.000,4.000,2.497,1.111,True
Year,0.000,1.000,0.501,0.500,True
Month,1.000,12.000,6.520,3.452,True
IsHoliday,0.000,1.000,0.029,0.167,True
Weekday,0.000,6.000,2.997,2.005,True
IsWorkday,0.000,1.000,0.684,0.465,True
Weather,1.000,3.000,1.395,0.545,True
Temp,0.059,0.862,0.495,0.183,False
FeelsLike,0.079,0.841,0.474,0.163,False


- The scales differ, which matters for SGD and regularization: the 4 decimal columns lie between 0 and 1, while the counts reach the thousands.
- `Casual` and `Registered` add up to `Rentals`, as two parts of the target would. This matters for whether they can be used as attributes.
- `Humidity` is the only decimal column with a minimum of 0.000, a value to check as a possible error.

7 of the 15 numeric columns are continuous: the 4 decimal columns (measurements) and the 3 rental counts, which are whole numbers but are amounts with 606 to 696 distinct values.

## Date

In [7]:
days_since_first = (df["Date"] - df["Date"].min()).dt.days

date_summary = pd.DataFrame(
	{
		"first date": [ df["Date"].min().date() ],
		"last date": [ df["Date"].max().date() ],
		"distinct dates": [ df["Date"].nunique() ],
		"days from first to last date": [ days_since_first.max() ],
		"rows 1 day after the previous row": [ (df["Date"].diff().dt.days == 1).sum() ],
		"rows where ID = days since first date + 1": [ (df["ID"] == days_since_first + 1).sum() ],
	},
	index=["Date"],
)

date_summary.T

,Date
first date,2011-01-01
last date,2012-12-31
distinct dates,731
days from first to last date,730
rows 1 day after the previous row,730
rows where ID = days since first date + 1,731


- The rows are in date order, one per day with no gaps (all 730 rows after the first are 1 day after the row before), which matters for how the data is split.
- Counted as days since the first date, `Date` is an amount from 0 to 730, so it is continuous.
- `ID` equals that day count plus 1 in all 731 rows. It numbers the rows, so it is neither continuous nor categorical, and it carries the same information as `Date`.

**8 attributes are continuous: the 4 measurements (`Temp`, `FeelsLike`, `Humidity`, and `Wind`), the 3 rental counts (`Casual`, `Registered`, and the target `Rentals`), and `Date`.**

# 3. Which Attributes are Categorical?

An attribute is categorical if its values name a group that a row belongs to: the value is a label, not an amount. The candidates are the 7 columns with 12 or fewer distinct values from section 2.

In [8]:
categorical_cols = ["Season", "Year", "Month", "IsHoliday", "Weekday", "IsWorkday", "Weather"]

category_levels = pd.DataFrame({
	"distinct values": df[categorical_cols].nunique(),
	"values": [", ".join(map(str, sorted(df[col].unique()))) for col in categorical_cols],
	"fewest rows per value": [df[col].value_counts().min() for col in categorical_cols],
	"most rows per value": [df[col].value_counts().max() for col in categorical_cols],
})

category_levels

,distinct values,values,fewest rows per value,most rows per value
Season,4,"1, 2, 3, 4",178,188
Year,2,"0, 1",365,366
Month,12,"1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12",57,62
IsHoliday,2,"0, 1",21,710
Weekday,7,"0, 1, 2, 3, 4, 5, 6",104,105
IsWorkday,2,"0, 1",231,500
Weather,3,"1, 2, 3",21,463


- All 7 take a few whole-number values, and their names describe groups of days, so the numbers are labels, not amounts. They are stored as integer codes, which matters for how they are encoded.
- `IsHoliday` and `Weather` each have a value with only 21 of the 731 rows, which matters for any split of the rows. `Season`, `Year`, `Month`, and `Weekday` are close to evenly filled.

**7 attributes are categorical: `Season`, `Year`, `Month`, `IsHoliday`, `Weekday`, `IsWorkday`, and `Weather`.**

**1 column is neither continuous nor categorical: `ID`, the row number (see the Date section).**